# 03 — Allocation diagnostic

Run all five diagnostic sections (combined-book exposure, correlation,
redundancy, risk contribution, benchmark comparison) on the resolved book
and export `reports/allocation_diagnostic.html`.


In [ ]:
from datetime import date
from pathlib import Path
import warnings

import pandas as pd

from hailmary.allocation.diagnostic import (
    benchmark_comparison, combined_exposure, combined_exposure_figure,
    correlation_figure, correlation_matrix, redundancy_pairs,
    render_html_report, risk_contribution,
)
from hailmary.allocation.holdings_book import load_book
from hailmary.allocation.portfolios import Role
from hailmary.allocation.reconcile import render_reconcile_report
from hailmary.allocation.returns import last_business_day_on_or_before
from hailmary.data.providers import YahooFinanceProvider

HOLDING_XLSX = Path("../../data/holding.xlsx")
REPORT_PATH = Path("../../reports/allocation_diagnostic.html")
RECONCILE_PATH = Path("../../reports/reconcile.html")
AS_OF = date(2026, 5, 31)
START = date(2022, 1, 1)
END = last_business_day_on_or_before(date.today())
REDUNDANCY_THRESHOLD = 0.85

ALIGN_WINDOW = True
TARGET_ANN_RETURN = 0.05
RECONCILE_AS_OF = END

print(f"as_of={AS_OF}  window={START}..{END}  align_window={ALIGN_WINDOW}  "
      f"target_ann_return={TARGET_ANN_RETURN:.1%}  reconcile_as_of={RECONCILE_AS_OF}")


## Load book + fetch returns

In [ ]:
portfolios = load_book(HOLDING_XLSX, as_of=AS_OF)
holding = [p for p in portfolios if Role.HOLDING in p.roles]
tickers = sorted({
    h.metadata.ticker for p in holding for h in p.holdings
    if not h.metadata.ticker.startswith('CASH_')
})
provider = YahooFinanceProvider()
returns = provider.get_returns(tickers, START, END)
print(f'Resolved {len(portfolios)} portfolios; fetched {returns.shape[1]} ticker series')


## Fetch USDSGD (daily series for return FX adjustment)

In [ ]:
fx_bars = provider.get_bars(['USDSGD=X'], START, END)
fx_series_usd_sgd = fx_bars.xs('USDSGD=X', level=0)['close']
# Statement-date FX is derived from the PortfolioValue (SGD)/(Base) ratio
# for any USD-base sleeve — that's the exact rate Stashaway applied. Yahoo
# daily series is still used for compounding return adjustments.
stashaway_fx = next(
    p.metadata['statement_fx_usd_sgd'] for p in portfolios
    if p.currency == 'USD' and 'statement_fx_usd_sgd' in p.metadata
)
yahoo_spot = float(fx_series_usd_sgd.iloc[-1])
print('Sheet-derived statement-date FX (used for AUM):  1 USD = {:.4f} SGD'.format(stashaway_fx))
print('Yahoo USDSGD spot ({}, used for daily series): 1 USD = {:.4f} SGD'.format(
    fx_series_usd_sgd.index.max().date(), yahoo_spot,
))


## Validation — no portfolio silently dropped + window summary

In [ ]:
from hailmary.allocation.diagnostic import _build_returns_panel, PortfolioDroppedError
try:
    _validation_panel = _build_returns_panel(
        holding, returns=returns, fx_series_usd_sgd=fx_series_usd_sgd, strict=True,
    )
    print(f'All {len(holding)} HOLDING portfolios resolved cleanly '
          f'({_validation_panel.shape[1]} series, {_validation_panel.shape[0]:,} dates).')
except PortfolioDroppedError as exc:
    print('FAIL — would drop portfolios:')
    for name, reason in exc.dropped:
        print(f'  {name}: {reason}')
    raise

first_dates = (
    _validation_panel.apply(lambda c: c.dropna().index.min().date())
    .sort_values(ascending=False)
)
common_start = first_dates.iloc[0]
aligned_days = len(_validation_panel.loc[str(common_start):].dropna(how='any'))
print()
print(f'Common-history window: [{common_start}..{END}] ({aligned_days:,} aligned dates).')
print('Per-portfolio first-data dates (latest first — these are what shrink the window):')
for name, first_date in first_dates.head(8).items():
    marker = '  ← constrains common_start' if first_date == common_start else ''
    print(f'  {name:<22} {first_date}{marker}')
if len(first_dates) > 8:
    remaining = first_dates.iloc[8:]
    print(f'  ...{len(remaining)} more portfolios start between '
          f'{remaining.min()} and {remaining.max()}')
print()
print('Set ALIGN_WINDOW=True (default) → combined-book metrics use this aligned window.')
print('Set ALIGN_WINDOW=False → full per-portfolio histories with dynamic-renorm weights.')

## Combined-book exposure

In [ ]:
exposure = combined_exposure(portfolios)
for dim, df in exposure.items():
    print(f'\n--- {dim.replace("_", " ").title()} ---')
    display(df)
combined_exposure_figure(exposure)

## Correlation matrix

In [ ]:
corr = correlation_matrix(portfolios, returns=returns, fx_series_usd_sgd=fx_series_usd_sgd)
display(corr.round(3))
correlation_figure(corr)

## Redundancy (threshold default 0.85)

In [ ]:
pairs = redundancy_pairs(corr, threshold=REDUNDANCY_THRESHOLD, portfolios=portfolios)
if pairs:
    pd.DataFrame(pairs, columns=['a', 'b', 'rho', 'candidate'])
else:
    print(f'No portfolio pairs above ρ = {REDUNDANCY_THRESHOLD}.')
    print('If your customs are uncorrelated by design, this is expected — drop the threshold to 0.7 to surface near-redundancy.')

## Risk contribution

In [ ]:
risk = risk_contribution(portfolios, returns=returns)
print('--- By portfolio ---')
display(risk['by_portfolio'].round(4))
print('--- By holding (top 15) ---')
display(risk['by_holding'].head(15).round(4))

## Benchmark comparison

In [ ]:
bench = benchmark_comparison(portfolios, returns=returns, fx_series_usd_sgd=fx_series_usd_sgd)
bench.round(3)

## Export HTML report (strict — fails if any portfolio would drop)

In [ ]:
out = render_html_report(
    portfolios,
    REPORT_PATH,
    returns=returns,
    redundancy_threshold=REDUNDANCY_THRESHOLD,
    fx_series_usd_sgd=fx_series_usd_sgd,
    align_window=ALIGN_WINDOW,
    target_ann_return=TARGET_ANN_RETURN,
    reconciliation_as_of=RECONCILE_AS_OF,
    links_path=HOLDING_XLSX,
    title="Stashaway book — allocation diagnostic",
)
print(f"Wrote {out.resolve()}")


## Export reconcile.html

The reconcile report answers ONE question: does our model match the app?
App start/end values come from the `PortfolioValue (SGD)` columns;
deposits from the `Deposits` sheet; our model is the validation column.


In [ ]:
out_reconcile = render_reconcile_report(
    portfolios,
    RECONCILE_PATH,
    end=END,
    price_source=provider,
    fx_series_usd_sgd=fx_series_usd_sgd,
    links_path=HOLDING_XLSX,
)
print(f'Wrote {out_reconcile.resolve()}')
